In [1]:
!pip install -q transformers==4.46.3 sentencepiece protobuf datasets evaluate peft accelerate hf_transfer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 61.1 MB/s eta 0:00:00


In [2]:
import glob, os

D = os.path.dirname(glob.glob("/kaggle/input/**/train.py", recursive=True)[0])
print(f"Dataset path located at: {D}")

Dataset path located at: /kaggle/input/datasets/marwanelamami13/bayan-model2-data


In [3]:
!cd {D} && CUDA_VISIBLE_DEVICES=0 python train.py \
    --run model2-arabart \
    --model moussaKam/AraBART \
    --prefix "بسط: " \
    --lr 5e-5 \
    --train train.jsonl \
    --select select_dev.jsonl \
    --evals test_*.jsonl \
    --out /kaggle/working

config.json: 1.39kB [00:00, 5.92MB/s]
sentencepiece.bpe.model: 100%|██████████████| 1.32M/1.32M [00:01<00:00, 946kB/s]
pytorch_model.bin: 100%|██████████████████████| 557M/557M [00:04<00:00, 121MB/s]
[model2-arabart       8s] model moussaKam/AraBART 139M params
[model2-arabart      11s] train 22733  select 900
[model2-arabart      11s] 1421 steps/epoch, eval every 710
{'loss': 3.0291, 'grad_norm': 12.655401229858398, 'learning_rate': 2.3000000000000003e-05, 'epoch': 0.04}
{'loss': 1.7995, 'grad_norm': 8.333952903747559, 'learning_rate': 4.8e-05, 'epoch': 0.07}
{'loss': 1.6023, 'grad_norm': 6.791831016540527, 'learning_rate': 4.983699503897945e-05, 'epoch': 0.11}
{'loss': 1.5172, 'grad_norm': 6.623223781585693, 'learning_rate': 4.965981573352232e-05, 'epoch': 0.14}
{'loss': 1.5411, 'grad_norm': 6.177732467651367, 'learning_rate': 4.94826364280652e-05, 'epoch': 0.18}
{'loss': 1.4201, 'grad_norm': 7.8463616371154785, 'learning_rate': 4.930545712260809e-05, 'epoch': 0.21}
{'loss': 1.4481, 

In [4]:
import os
import glob
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tok_hf = UserSecretsClient().get_secret("HF_TOKEN")

# Put your actual personal Hugging Face username here:
hf_username = "YOUR_HF_USERNAME" 
repo_id = f"{hf_username}/bayan-model2-arabart"
import os
import glob
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Retrieve token from secrets
tok_hf = UserSecretsClient().get_secret("HF_TOKEN")

# Personal Hugging Face repository target
repo_id = "SanadAli/bayan-model2-arabart"

# 1. Locate trained weights checkpoint
weight_files = glob.glob("/kaggle/working/**/model.safetensors", recursive=True) + \
               glob.glob("/kaggle/working/**/pytorch_model.bin", recursive=True)

if not weight_files:
    raise FileNotFoundError("Could not find model weight files in /kaggle/working.")

target_dir = None
for f in weight_files:
    if "checkpoint-12780" in f:
        target_dir = os.path.dirname(f)
        break

if not target_dir:
    target_dir = os.path.dirname(weight_files[0])

print(f"Found trained weights at: {target_dir}")

# 2. Load model & base tokenizer
print("Loading model and tokenizer...")
model = AutoModelForSeq2SeqLM.from_pretrained(target_dir)
tokenizer = AutoTokenizer.from_pretrained("moussaKam/AraBART")

# 3. Upload to Hugging Face
print(f"Uploading model files to https://huggingface.co/{repo_id}...")
tokenizer.push_to_hub(repo_id, private=True, token=tok_hf)
model.push_to_hub(repo_id, private=True, token=tok_hf)

print(f"✅ Upload complete! Check your repository at: https://huggingface.co/{repo_id}")

Found trained weights at: /kaggle/working/model2-arabart/model
Loading model and tokenizer...
Uploading model files to https://huggingface.co/SanadAli/bayan-model2-arabart...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Upload complete! Check your repository at: https://huggingface.co/SanadAli/bayan-model2-arabart
